# Live LLM Arena with Eden AI

Send the same prompt to four LLMs in parallel, watch them stream side-by-side, then pick the winner (human vote or LLM-as-judge).

Same API, same key, same response format — only the `model` string changes per call. That is the point of this notebook.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var).

In [ ]:
%pip install --quiet aiohttp ipywidgets nest_asyncio

## 1. Configuration

Each entry is one model in the arena. Add or remove rows freely — the grid auto-resizes.

In [ ]:
import os

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY", "YOUR_EDEN_AI_API_KEY")
EDENAI_URL = "https://api.edenai.run/v3/llm/chat/completions"

MODELS = [
    {"label": "Claude",   "model": "anthropic/claude-sonnet-4-5"},
    {"label": "GPT",      "model": "openai/gpt-4"},
    {"label": "DeepSeek", "model": "deepseek/deepseek-chat"},
    {"label": "GLM",      "model": "zai/glm-5.1"},
]

JUDGE_MODEL = "anthropic/claude-sonnet-4-5"

## 2. Streaming caller

Eden AI's `/v3/llm/chat/completions` is OpenAI-compatible, including SSE streaming. Each chunk arrives as `data: { ... }` with the new token in `choices[0].delta.content`. We pipe that into an `ipywidgets.Output` so each model's pane fills in live.

In [ ]:
import json
import time
import aiohttp


async def stream_model(session, model_cfg, prompt, output_widget):
    headers = {
        "Authorization": f"Bearer {EDENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model_cfg["model"],
        "messages": [{"role": "user", "content": prompt}],
        "stream": True,
    }

    output_widget.clear_output()
    full_text = []
    first_token_at = None
    start = time.perf_counter()

    async with session.post(EDENAI_URL, headers=headers, json=payload) as resp:
        if resp.status != 200:
            with output_widget:
                print(f"[error {resp.status}] {await resp.text()}")
            return {"label": model_cfg["label"], "text": "", "first_token": None, "total": None}

        async for raw in resp.content:
            line = raw.decode("utf-8").strip()
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            try:
                chunk = json.loads(data)
            except json.JSONDecodeError:
                continue
            token = chunk.get("choices", [{}])[0].get("delta", {}).get("content", "")
            if not token:
                continue
            if first_token_at is None:
                first_token_at = time.perf_counter() - start
            full_text.append(token)
            with output_widget:
                print(token, end="")

    return {
        "label": model_cfg["label"],
        "text": "".join(full_text),
        "first_token": first_token_at,
        "total": time.perf_counter() - start,
    }

## 3. Arena UI

A 2-column grid of streaming panes plus a prompt box, fight button, and per-model vote buttons.

In [ ]:
from ipywidgets import (
    HBox, VBox, Output, Textarea, Button, Label, GridBox, Layout,
)
from IPython.display import display

prompt_box = Textarea(
    placeholder="Type a prompt for all models, then press Fight…",
    layout=Layout(width="100%", height="60px"),
)
fight_btn = Button(description="⚔  Fight", button_style="primary")

panel_layout = Layout(
    border="1px solid #ddd",
    padding="8px",
    height="220px",
    overflow="auto",
)
panels = [Output(layout=panel_layout) for _ in MODELS]
header_labels = [Label(value=m["label"]) for m in MODELS]
panel_blocks = [VBox([header_labels[i], panels[i]]) for i in range(len(MODELS))]
grid = GridBox(
    panel_blocks,
    layout=Layout(grid_template_columns="repeat(2, 1fr)", grid_gap="8px"),
)

vote_btns = [Button(description=f"Vote {m['label']}") for m in MODELS]
judge_btn = Button(description="🧑‍⚖️ LLM judge")
vote_bar = HBox([*vote_btns, judge_btn])

scoreboard = Output()
stats = Output()

display(VBox([prompt_box, fight_btn, grid, stats, vote_bar, scoreboard]))

## 4. Wire it up

Fight button → fan out to all models in parallel via `asyncio.gather`. Vote button increments a score; LLM-judge button sends every response to a referee model.

In [ ]:
import asyncio
import nest_asyncio
from IPython.display import clear_output

nest_asyncio.apply()

scores = {m["label"]: 0 for m in MODELS}
last_round = {}


def render_scoreboard():
    with scoreboard:
        clear_output()
        print("Scoreboard")
        print("----------")
        for label, n in sorted(scores.items(), key=lambda kv: -kv[1]):
            print(f"  {label}: {n}")


def render_stats(results):
    with stats:
        clear_output()
        print("Round timing (seconds)")
        for r in results:
            ft = f"{r['first_token']:.2f}" if r["first_token"] else "-"
            tt = f"{r['total']:.2f}" if r["total"] else "-"
            print(f"  {r['label']:<10}  first token: {ft}s   total: {tt}s")


async def run_round(prompt):
    async with aiohttp.ClientSession() as session:
        tasks = [
            stream_model(session, MODELS[i], prompt, panels[i])
            for i in range(len(MODELS))
        ]
        return await asyncio.gather(*tasks)


def on_fight(_):
    prompt = prompt_box.value.strip()
    if not prompt:
        return
    results = asyncio.run(run_round(prompt))
    last_round.clear()
    last_round.update({r["label"]: r["text"] for r in results})
    render_stats(results)


def make_vote_handler(label):
    def handler(_):
        scores[label] += 1
        render_scoreboard()
    return handler


async def call_judge(prompt, responses):
    blocks = "\n\n".join(f"### {label}\n{text}" for label, text in responses.items())
    judge_prompt = (
        f"Original question:\n{prompt}\n\n"
        f"Candidate answers:\n{blocks}\n\n"
        "Pick the single best answer for accuracy and clarity. "
        "Respond with the label only (one word)."
    )
    async with aiohttp.ClientSession() as session:
        headers = {
            "Authorization": f"Bearer {EDENAI_API_KEY}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": JUDGE_MODEL,
            "messages": [{"role": "user", "content": judge_prompt}],
        }
        async with session.post(EDENAI_URL, headers=headers, json=payload) as r:
            data = await r.json()
            return data["choices"][0]["message"]["content"].strip()


def on_judge(_):
    if not last_round:
        return
    winner = asyncio.run(call_judge(prompt_box.value.strip(), last_round))
    for label in scores:
        if label.lower() in winner.lower():
            scores[label] += 1
            break
    with scoreboard:
        clear_output()
        print(f"Judge picked: {winner}")
    render_scoreboard()


fight_btn.on_click(on_fight)
judge_btn.on_click(on_judge)
for btn, m in zip(vote_btns, MODELS):
    btn.on_click(make_vote_handler(m["label"]))

## 5. Try these prompts

- **Reasoning:** *A bat and a ball cost \$1.10 in total. The bat costs \$1 more than the ball. How much does the ball cost?*
- **Creative:** *Write a 4-line poem about a lighthouse, but every line must start with the letter S.*
- **Code:** *Write a Python one-liner that returns the longest word in a string, ties broken by leftmost.*
- **Multilingual:** *Translate “knowledge is power” into Japanese, then explain the literal meaning of each character.*

## 6. Customize

- **More models:** add rows to `MODELS` — the grid auto-expands. Try `google/gemini-2.5-flash`, `mistral/mistral-large`, `cohere/command-r-plus`.
- **Smart routing:** set `"model": "edenai/smart-route"` to let Eden AI pick the cheapest/fastest model that fits the prompt.
- **Persist scores:** dump `scores` to JSON between sessions to keep a running leaderboard.